In [1]:
from obspy.geodetics.base import gps2dist_azimuth
import obspy
import requests
import pandas as pd
token = '' # ONC token

### get all locationCoces for Broadband instruments at Endeavour

In [ ]:
url = 'https://data.oceannetworks.ca/api/locations?locationCode=END&deviceCategoryCode=BBS&includeChildren=True&token='+token
r = requests.get(url)
df = pd.DataFrame(r.json())
df ['type'] = 'BBS'

### get all locationCoces for short period instruments at Endeavour

In [ ]:
url = 'https://data.oceannetworks.ca/api/locations?locationCode=END&deviceCategoryCode=SPS&includeChildren=True&token='+token
r = requests.get(url)
df2 = pd.DataFrame(r.json())
df2['type']='SPS'

### get all locationCoces for accelerometer instruments at Endeavour

In [ ]:
url = 'https://data.oceannetworks.ca/api/locations?locationCode=END&deviceCategoryCode=Accelerometer&includeChildren=True&token='+token
r = requests.get(url)
df3 = pd.DataFrame(r.json())
df3['type']='ACCELEROMETER'
df3

### combine all

In [ ]:
df2 = pd.concat([df2,df])
df2 = pd.concat([df2,df3])

In [ ]:

sites= {'site':df2.locationCode.values, 'lat':df2.lat.values,'lon':df2.lon.values, 'type':df2.type.values}
sites

In [ ]:
len(sites['site'])

### calculate distances between each site

In [ ]:
from itertools import combinations

# Calculate ellipsoidal distance (meters), azimuth, and back-azimuth for all site pairs
site_names = sites['site']
lats = sites['lat']
lons = sites['lon']

for i, j in combinations(range(len(site_names)), 2):
    distance_m, azimuth, back_azimuth = gps2dist_azimuth(lats[i], lons[i], lats[j], lons[j])
    distance_km = distance_m / 1000.0
    print(f"Distance between {site_names[i]} and {site_names[j]}: {distance_km:.3f} km  |  Az: {azimuth:.2f}°  |  Back-Az: {back_azimuth:.2f}°")


### get seismometer deployments between May 2022 and Jan 2023

In [ ]:
deployments = []

for i in range(len(sites['site'])):
    location_code = sites['site'][i]
    device_category = sites['type'][i]
    url = (
        f"https://data.oceannetworks.ca/api/deployments"
        f"?locationCode={location_code}"
        f"&deviceCategoryCode={device_category}"
        f"&dateFrom=2022-05-01"
        f"&dateTo=2023-01-31"
        f"&token={token}"
    )
    r = requests.get(url)
    results = r.json()
    if isinstance(results, list) and len(results) > 0:
        for rec in results:
            rec['queryLocationCode'] = location_code
            rec['queryDeviceCategory'] = device_category
        deployments.extend(results)
        print(f"{location_code} ({device_category}): {len(results)} deployment(s)")
    else:
        print(f"{location_code} ({device_category}): no deployments found")

df_deployments = pd.DataFrame(deployments)
df_deployments


## Find the data availability in Earthscope

In [16]:
url = 'https://service.iris.edu/fdsnws/availability/1/extent?format=text&net=NV&cha=*H*&orderby=nslc_time_quality_samplerate&includerestricted=true&nodata=404'

In [17]:
r = requests.get(url)
r.content

b'#Network Station Location Channel Quality SampleRate Earliest Latest Updated TimeSpans Restriction\nNV BACME W1 HNE M 200.0 2018-06-11T00:00:00.000000Z 2026-06-10T15:45:18.700000Z 2026-06-11T08:40:46Z 1021 OPEN\nNV BACME W1 HNN M 200.0 2018-06-11T00:00:00.000000Z 2026-06-10T15:45:18.850000Z 2026-06-11T08:40:46Z 977 OPEN\nNV BACME W1 HNZ M 200.0 2018-06-11T00:00:00.000000Z 2026-06-10T15:45:18.620000Z 2026-06-11T08:40:46Z 945 OPEN\nNV BACND Z1 AHD M 1079.99 2019-06-12T00:00:00.000600Z 2024-06-23T23:59:59.999111Z 2024-06-25T01:31:17Z 6509 OPEN\nNV BACND Z1 AHD M 1080.0 2018-08-16T12:27:04.186000Z 2025-04-12T23:59:59.999544Z 2025-04-15T19:43:01Z 880003 OPEN\nNV BACND Z1 AHD M 1080.01 2020-01-04T00:00:00.000900Z 2024-04-11T23:59:59.999296Z 2024-04-15T17:28:36Z 3004 OPEN\nNV BACND Z1 AHD M 1080.03 2019-03-21T00:00:00.000300Z 2021-04-06T22:28:29.977089Z 2021-04-07T09:05:31Z 1439 OPEN\nNV CBC27 -- CH1 M 499.977 2026-02-03T00:00:00.000000Z 2026-02-03T23:59:59.998000Z 2026-04-30T19:41:42Z 1 RE

In [18]:
import io

lines = r.content.decode('utf-8').splitlines()
header = [l for l in lines if l.startswith('#')]
data_lines = [l for l in lines if l and not l.startswith('#')]

columns = header[-1].lstrip('#').split()
df_availability = pd.read_csv(
    io.StringIO('\n'.join(data_lines)),
    sep=r'\s+',
    names=columns,
    engine='python'
)
df_availability


,Network,Station,Location,Channel,Quality,SampleRate,Earliest,Latest,Updated,TimeSpans,Restriction
0,NV,BACME,W1,HNE,M,200.00,2018-06-11T00:00:00.000000Z,2026-06-10T15:45:18.700000Z,2026-06-11T08:40:46Z,1021,OPEN
1,NV,BACME,W1,HNN,M,200.00,2018-06-11T00:00:00.000000Z,2026-06-10T15:45:18.850000Z,2026-06-11T08:40:46Z,977,OPEN
2,NV,BACME,W1,HNZ,M,200.00,2018-06-11T00:00:00.000000Z,2026-06-10T15:45:18.620000Z,2026-06-11T08:40:46Z,945,OPEN
3,NV,BACND,Z1,AHD,M,1079.99,2019-06-12T00:00:00.000600Z,2024-06-23T23:59:59.999111Z,2024-06-25T01:31:17Z,6509,OPEN
4,NV,BACND,Z1,AHD,M,1080.00,2018-08-16T12:27:04.186000Z,2025-04-12T23:59:59.999544Z,2025-04-15T19:43:01Z,880003,OPEN
...,...,...,...,...,...,...,...,...,...,...,...
410,NV,NSMTC,G1,CHN,M,500.00,2017-08-20T00:00:00.000000Z,2026-03-04T19:24:11.938000Z,2026-03-11T20:55:16Z,847,OPEN
411,NV,NSMTC,G1,CHZ,M,500.00,2017-08-20T00:00:00.000000Z,2026-03-04T19:24:11.938000Z,2026-03-11T20:55:16Z,743,OPEN
412,NV,NSMTC,G2,CH1,M,500.00,2017-08-20T00:00:00.000000Z,2024-03-12T17:56:18.224000Z,2024-09-11T23:18:39Z,1409,OPEN
413,NV,NSMTC,G2,CH2,M,500.00,2017-08-20T00:00:00.000000Z,2024-03-12T17:56:36.480000Z,2024-09-11T23:18:39Z,1137,OPEN


In [33]:
selected_channels = ['EHZ', 'HHZ', 'CHZ', 'CH3', 'HH3', 'HN3', 'HNZ','CNZ','CN3']
df_filtered = df_availability[df_availability['Channel'].isin(selected_channels)]
df_filtered


,Network,Station,Location,Channel,Quality,SampleRate,Earliest,Latest,Updated,TimeSpans,Restriction
2,NV,BACME,W1,HNZ,M,200.0000,2018-06-11T00:00:00.000000Z,2026-06-10T15:45:18.620000Z,2026-06-11T08:40:46Z,945,OPEN
23,NV,CBC27,--,CH3,M,499.9770,2025-11-01T00:00:00.000000Z,2026-04-27T23:41:07.262000Z,2026-04-29T19:41:54Z,3,RESTRICTED
24,NV,CBC27,--,CH3,M,499.9960,2025-12-01T00:00:00.000000Z,2026-05-25T21:37:56.386000Z,2026-05-26T08:41:47Z,3,RESTRICTED
25,NV,CBC27,--,CH3,M,499.9980,2025-11-04T00:00:00.000000Z,2025-11-04T23:59:59.998000Z,2025-11-25T20:42:19Z,1,RESTRICTED
26,NV,CBC27,--,CH3,M,499.9990,2025-09-07T00:00:00.000000Z,2026-04-26T23:59:59.998000Z,2026-04-30T19:41:32Z,8,RESTRICTED
...,...,...,...,...,...,...,...,...,...,...,...
400,NV,NCHR,--,EHZ,M,199.9990,2016-08-17T00:00:00.000000Z,2025-03-04T23:59:59.995000Z,2025-03-05T09:46:01Z,18,OPEN
401,NV,NCHR,--,EHZ,M,200.0000,2016-06-20T16:36:14.840000Z,2026-06-10T15:45:20.590000Z,2026-06-11T08:41:00Z,1428,OPEN
402,NV,NCHR,--,EHZ,M,99.9999,2010-12-02T00:00:00.000000Z,2012-07-04T23:59:59.990000Z,2017-03-25T07:40:53Z,6,OPEN
411,NV,NSMTC,G1,CHZ,M,500.0000,2017-08-20T00:00:00.000000Z,2026-03-04T19:24:11.938000Z,2026-03-11T20:55:16Z,743,OPEN


In [1]:
import plotly.express as px

plot_df = df_filtered.copy()
plot_df = plot_df.sort_values(by='Station')
plot_df['Earliest'] = pd.to_datetime(plot_df['Earliest'])
plot_df['Latest'] = pd.to_datetime(plot_df['Latest'])
plot_df['label'] = plot_df['Station'] + '.' +plot_df['Location']+'.'+ plot_df['Channel']

fig = px.timeline(
    plot_df,
    x_start='Earliest',
    x_end='Latest',
    y='label',
    color='Channel',
    title='Data Availability — NV Network',
    labels={'label': 'Station.Location.Channel'},
    width=1400,
    height=800,
)
fig.update_yaxes(autorange='reversed')
fig.show()


NameError: name 'df_filtered' is not defined